# Estadificación del sueño — validación cruzada por sujeto

Notebook autocontenido. Ejecuta la batería completa de experimentos sobre GPU:

1. **Validación cruzada por persona** (k folds) del modelo principal
2. **Variantes** base / small / tiny
3. **Ablación**: sin contexto, sin SE, sin atención, convolución estándar

Reglas metodológicas que el código impone y que no hay que relajar:

- Los folds se hacen **por persona**, no por grabación. En Sleep-EDF `SC4001` y `SC4002` son la misma persona en dos noches; separarlas mete al mismo individuo en train y test.
- Las ventanas de secuencia **nunca cruzan** el límite entre grabaciones.
- En evaluación cada epoch se puntúa **exactamente una vez**.

**Antes de ejecutar:** Entorno de ejecución → Cambiar tipo de entorno → GPU.

In [ ]:
import torch, os
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('dispositivo:', torch.cuda.get_device_name(0))
    print('memoria: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print('SIN GPU. Entorno de ejecucion -> Cambiar tipo de entorno -> GPU')
print('torch', torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# AJUSTA esta ruta a donde hayas subido la carpeta data/colab
DATA_DIR = '/content/drive/MyDrive/sleepedf'
OUT_DIR  = '/content/drive/MyDrive/sleepedf/resultados'

import os, json
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DATA_DIR))
print(json.load(open(os.path.join(DATA_DIR, 'dataset_info.json')))['protocol'])

In [ ]:
import numpy as np, time

t0 = time.time()
Z = np.load(os.path.join(DATA_DIR, 'sleepedf_all_fp16.npz'), allow_pickle=True)
X_all = Z['X']            # (N, 3000) float16
y_all = Z['y'].astype(np.int64)
subject = Z['subject']    # id de grabacion, p.ej. 'SC4001'
person  = Z['person']     # id de persona,  p.ej. '00'
print('cargado en %.0f s' % (time.time()-t0))
print('X', X_all.shape, X_all.dtype)
print('epochs {:,} | grabaciones {} | personas {}'.format(
    len(y_all), len(set(subject.tolist())), len(set(person.tolist()))))

# comprobacion de contigüidad: imprescindible para las ventanas
changes = int((subject[1:] != subject[:-1]).sum()) + 1
assert changes == len(set(subject.tolist())), 'grabaciones no contiguas'
print('bloques contiguos por grabacion: OK')

## Modelo

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class DSConv1d(nn.Module):
    def __init__(s, i, o, k, stride=1, padding=0):
        super().__init__()
        s.dw = nn.Conv1d(i, i, k, stride=stride, padding=padding, groups=i, bias=False)
        s.pw = nn.Conv1d(i, o, 1, bias=False)
        s.bn = nn.BatchNorm1d(o)
    def forward(s, x): return F.relu(s.bn(s.pw(s.dw(x))))

class StdConv1d(nn.Module):
    def __init__(s, i, o, k, stride=1, padding=0):
        super().__init__()
        s.c = nn.Conv1d(i, o, k, stride=stride, padding=padding, bias=False)
        s.bn = nn.BatchNorm1d(o)
    def forward(s, x): return F.relu(s.bn(s.c(x)))

class SE(nn.Module):
    def __init__(s, c, r=16):
        super().__init__()
        s.f = nn.Sequential(nn.Linear(c, max(c//r,1), bias=False), nn.ReLU(True),
                            nn.Linear(max(c//r,1), c, bias=False), nn.Sigmoid())
    def forward(s, x):
        b, c, _ = x.shape
        return x * s.f(F.adaptive_avg_pool1d(x,1).view(b,c)).view(b,c,1)

class LTA(nn.Module):
    """Atencion temporal ligera dentro del epoch."""
    def __init__(s, c, heads=4):
        super().__init__()
        s.h, s.d = heads, c//heads
        s.scale = s.d ** -0.5
        s.qkv = nn.Linear(c, c*3, bias=False)
        s.proj = nn.Linear(c, c, bias=False)
        s.norm = nn.LayerNorm(c)
    def forward(s, x):
        x = x.transpose(1,2); B,T,C = x.shape
        r = x; x = s.norm(x)
        qkv = s.qkv(x).reshape(B,T,3,s.h,s.d).permute(2,0,3,1,4)
        q,k,v = qkv[0],qkv[1],qkv[2]
        a = F.softmax((q @ k.transpose(-2,-1))*s.scale, dim=-1)
        x = (a @ v).transpose(1,2).reshape(B,T,C)
        return (s.proj(x)+r).transpose(1,2)

class Block(nn.Module):
    def __init__(s, i, o, k=7, use_se=True, use_attn=True, use_ds=True):
        super().__init__()
        s.conv = (DSConv1d if use_ds else StdConv1d)(i, o, k, padding=k//2)
        s.se = SE(o) if use_se else nn.Identity()
        s.at = LTA(o) if use_attn else nn.Identity()
        s.skip = nn.Conv1d(i,o,1) if i!=o else nn.Identity()
    def forward(s, x): return s.at(s.se(s.conv(x))) + s.skip(x)

BASE_FILTERS = {'base':32, 'small':24, 'tiny':16}

class EpochEncoder(nn.Module):
    def __init__(s, variant='base', use_se=True, use_attn=True, use_ds=True):
        super().__init__()
        f = BASE_FILTERS[variant]
        s.stem = nn.Sequential(nn.Conv1d(1,f,49,stride=4,padding=24),
                               nn.BatchNorm1d(f), nn.ReLU(True),
                               nn.MaxPool1d(4,4))
        kw = dict(use_se=use_se, use_attn=use_attn, use_ds=use_ds)
        s.blocks = nn.ModuleList([
            Block(f, f*2, **kw), nn.MaxPool1d(2,2),
            Block(f*2, f*4, **kw), nn.MaxPool1d(2,2),
            Block(f*4, f*4, **kw), nn.MaxPool1d(2,2),
            Block(f*4, f*4, **kw)])
        s.out_dim = f*4
    def forward(s, x):
        x = s.stem(x)
        for b in s.blocks: x = b(x)
        return F.adaptive_avg_pool1d(x,1).flatten(1)

class SequenceSleepNet(nn.Module):
    def __init__(s, variant='base', seq_encoder='gru', seq_len=21, hidden=64,
                 dropout=0.3, n_classes=5, use_se=True, use_attn=True, use_ds=True):
        super().__init__()
        s.kind = seq_encoder
        s.enc = EpochEncoder(variant, use_se, use_attn, use_ds)
        d = s.enc.out_dim
        if seq_encoder == 'gru':
            s.seq = nn.GRU(d, hidden, batch_first=True, bidirectional=True); ctx = hidden*2
        elif seq_encoder == 'attn':
            s.pos = nn.Parameter(torch.zeros(1, seq_len, d))
            nn.init.trunc_normal_(s.pos, std=0.02)
            s.norm = nn.LayerNorm(d)
            s.mha = nn.MultiheadAttention(d, 4, batch_first=True); ctx = d
        else:
            ctx = d
        s.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(ctx, hidden),
                               nn.ReLU(True), nn.Dropout(dropout),
                               nn.Linear(hidden, n_classes))
    def forward(s, x):
        B,L,T = x.shape
        e = s.enc(x.reshape(B*L,1,T)).reshape(B,L,-1)
        if s.kind == 'gru': c,_ = s.seq(e)
        elif s.kind == 'attn':
            h = s.norm(e + s.pos[:,:L]); a,_ = s.mha(h,h,h,need_weights=False); c = e+a
        else: c = e
        return s.head(c)
    def n_params(s): return sum(p.numel() for p in s.parameters() if p.requires_grad)
    def n_params_encoder(s): return sum(p.numel() for p in s.enc.parameters() if p.requires_grad)

for v in ['base','small','tiny']:
    for se in ['none','gru','attn']:
        m = SequenceSleepNet(v, se)
        print('%-6s %-5s total %8d  encoder %8d' % (v, se, m.n_params(), m.n_params_encoder()))

## Ventanas y evaluación

In [ ]:
from torch.utils.data import Dataset, DataLoader

def bounds_of(subj_arr, idx):
    """Bloques contiguos de la misma grabacion dentro de idx (ya ordenado)."""
    s = subj_arr[idx]
    ch = np.flatnonzero(s[1:] != s[:-1]) + 1
    st = np.concatenate(([0], ch)); en = np.concatenate((ch, [len(idx)]))
    return [(idx[a], idx[b-1]+1) for a,b in zip(st, en)]

class SeqDS(Dataset):
    """mode='train': ventanas solapadas. mode='eval': cada epoch se puntua una vez."""
    def __init__(s, X, y, bounds, L, mode, stride=1, augment=False):
        s.X, s.y, s.L, s.mode, s.aug = X, y, L, mode, augment
        s.items = []
        for a,b in bounds:
            if b-a < L: continue
            if mode=='train':
                s.items += [(p,0) for p in range(a, b-L+1, stride)]
            else:
                p = a
                while p+L <= b: s.items.append((p,0)); p += L
                if p < b: s.items.append((b-L, p-(b-L)))
        s.dropped = sum(b-a for a,b in bounds if b-a < L)
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        p, vf = s.items[i]
        x = torch.from_numpy(s.X[p:p+s.L].astype(np.float32))
        y = torch.from_numpy(s.y[p:p+s.L])
        if s.aug:
            x = x * (torch.rand(1).item()*0.45 + 0.8)
            x = x + torch.randn_like(x)*0.05
            if torch.rand(1).item() < 0.5: x = -x
        m = torch.ones(s.L, dtype=torch.bool)
        if vf: m[:vf] = False
        return x, y, m

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             confusion_matrix)

DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STAGES = ['W','N1','N2','N3','REM']

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, T = [], []
    for x,y,m in loader:
        x = x.to(DEV, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16, enabled=DEV.type=='cuda'):
            lg = model(x)
        p = lg.float().argmax(-1).cpu()
        P.append(p[m].numpy()); T.append(y[m].numpy())
    p, t = np.concatenate(P), np.concatenate(T)
    return dict(acc=accuracy_score(t,p), kappa=cohen_kappa_score(t,p),
                f1=f1_score(t,p,average='macro'),
                f1_class={n: float(v) for n,v in zip(
                    STAGES, f1_score(t,p,average=None,labels=range(5)))},
                cm=confusion_matrix(t,p,labels=range(5)).tolist(),
                n=int(len(t)),
                majority=float(np.bincount(t,minlength=5).max()/len(t)))

def run(train_idx, val_idx, variant='base', seq_encoder='gru', L=21,
        stride=7, epochs=12, bs=64, lr=1e-3, wd=1e-3, dropout=0.3,
        augment=True, seed=42, verbose=True, **abl):
    torch.manual_seed(seed); np.random.seed(seed)
    tr = SeqDS(X_all, y_all, bounds_of(subject, train_idx), L, 'train',
               stride=stride, augment=augment)
    va = SeqDS(X_all, y_all, bounds_of(subject, val_idx), L, 'eval')
    tl = DataLoader(tr, batch_size=bs, shuffle=True, num_workers=2,
                    pin_memory=True, drop_last=True)
    vl = DataLoader(va, batch_size=bs*2, shuffle=False, num_workers=2,
                    pin_memory=True)
    model = SequenceSleepNet(variant, seq_encoder, L, dropout=dropout, **abl).to(DEV)
    cnt = np.bincount(y_all[train_idx], minlength=5).astype(np.float64)
    w = torch.tensor(cnt.sum()/(5*np.maximum(cnt,1)), dtype=torch.float32, device=DEV)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=DEV.type=='cuda')
    best, best_state, hist = -1, None, []
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time()
        for x,y,m in tl:
            x,y = x.to(DEV,non_blocking=True), y.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16, enabled=DEV.type=='cuda'):
                loss = crit(model(x).reshape(-1,5), y.reshape(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        sch.step()
        r = evaluate(model, vl); r['epoch']=ep; r['sec']=time.time()-t0
        hist.append({k:r[k] for k in ('epoch','acc','kappa','f1','sec')})
        if r['kappa'] > best:
            best = r['kappa']
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_r = r
        if verbose:
            print('  ep %2d/%d  acc %.4f  kappa %.4f  f1 %.4f  (%.0fs)%s' %
                  (ep, epochs, r['acc'], r['kappa'], r['f1'], r['sec'],
                   '  <- mejor' if r['kappa']==best else ''))
    model.load_state_dict(best_state)
    best_r['n_parameters'] = model.n_params()
    best_r['n_parameters_encoder'] = model.n_params_encoder()
    best_r['history'] = hist
    best_r['dropped_epochs'] = va.dropped
    return best_r

## 1. Validación cruzada por persona

Los folds agrupan por **persona**, de modo que las dos noches de un sujeto caen siempre en el mismo lado. Esto es lo que da los intervalos de confianza que un revisor va a exigir.

In [ ]:
from sklearn.model_selection import GroupKFold

K = 5          # sube a 10 si el tiempo lo permite
EPOCHS = 12

gkf = GroupKFold(n_splits=K)
all_idx = np.arange(len(y_all))
folds = list(gkf.split(all_idx, y_all, groups=person))
for i,(tr_i,va_i) in enumerate(folds):
    print('fold %d: %6d train / %6d test epochs | %2d personas en test' %
          (i, len(tr_i), len(va_i), len(set(person[va_i].tolist()))))

cv = []
for i,(tr_i,va_i) in enumerate(folds):
    print('\n=== fold %d/%d ===' % (i+1, K))
    r = run(tr_i, va_i, variant='base', seq_encoder='gru', epochs=EPOCHS)
    r['fold'] = i
    cv.append(r)
    print('  fold %d -> acc %.4f  kappa %.4f  f1 %.4f' % (i, r['acc'], r['kappa'], r['f1']))
    json.dump(cv, open(os.path.join(OUT_DIR, 'cv_base_gru.json'), 'w'), indent=2)

k = np.array([r['kappa'] for r in cv]); a = np.array([r['acc'] for r in cv])
f = np.array([r['f1'] for r in cv])
print('\n===== VALIDACION CRUZADA (%d folds) =====' % K)
print('accuracy  %.4f +/- %.4f   [%.4f, %.4f]' % (a.mean(), a.std(), a.min(), a.max()))
print('kappa     %.4f +/- %.4f   [%.4f, %.4f]' % (k.mean(), k.std(), k.min(), k.max()))
print('macro-F1  %.4f +/- %.4f' % (f.mean(), f.std()))

## 2. Variantes y codificadores de secuencia

Tabla 1 del paper. Un solo fold para no multiplicar el coste; la incertidumbre la aporta la sección anterior.

In [ ]:
tr_i, va_i = folds[0]
tabla1 = {}
for variant in ['base','small','tiny']:
    for se in ['none','gru','attn']:
        name = '%s_%s' % (variant, se)
        print('\n=== %s ===' % name)
        r = run(tr_i, va_i, variant=variant, seq_encoder=se, epochs=EPOCHS, verbose=False)
        tabla1[name] = r
        print('  %-12s params %7d (enc %7d)  acc %.4f  kappa %.4f  f1 %.4f' %
              (name, r['n_parameters'], r['n_parameters_encoder'], r['acc'], r['kappa'], r['f1']))
        json.dump(tabla1, open(os.path.join(OUT_DIR,'tabla1_variantes.json'),'w'), indent=2)

print('\n%-12s %9s %9s %8s %8s' % ('config','params','encoder','acc','kappa'))
for n,r in tabla1.items():
    print('%-12s %9d %9d %8.4f %8.4f' % (n, r['n_parameters'], r['n_parameters_encoder'], r['acc'], r['kappa']))

## 3. Ablación

Tabla 2. Cada fila cambia **un solo** componente respecto al modelo completo.

In [ ]:
ABL = [('completo',            dict()),
       ('sin SE',              dict(use_se=False)),
       ('sin atencion intra',  dict(use_attn=False)),
       ('conv estandar',       dict(use_ds=False))]

tabla2 = {}
for label, kw in ABL:
    print('\n=== %s ===' % label)
    r = run(tr_i, va_i, variant='base', seq_encoder='gru', epochs=EPOCHS,
            verbose=False, **kw)
    tabla2[label] = r
    print('  %-20s params %7d  acc %.4f  kappa %.4f' %
          (label, r['n_parameters'], r['acc'], r['kappa']))
    json.dump(tabla2, open(os.path.join(OUT_DIR,'tabla2_ablacion.json'),'w'), indent=2)

ref = tabla2['completo']['kappa']
print('\n%-20s %9s %8s %8s %9s' % ('configuracion','params','acc','kappa','delta'))
for n,r in tabla2.items():
    print('%-20s %9d %8.4f %8.4f %+9.4f' % (n, r['n_parameters'], r['acc'], r['kappa'], r['kappa']-ref))

In [ ]:
print('Resultados guardados en', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print('  ', f, '%.1f KB' % (os.path.getsize(os.path.join(OUT_DIR,f))/1024))
print('\nDescarga estos .json y pasalos a Claude Code para generar tablas y figuras.')